# K5 Action Sub-400 Archive

Characterize the K5 recoloring action from archived sub-400 K43 colorings before introducing a neural policy. Each action ranks monochromatic K5 targets, evaluates all 1,022 nonmonochromatic replacement patterns for the most promising targets, and applies the best objective-ranked macro action.

This notebook runs the same greedy K5 macro-action experiment as `K5_Action_Baseline.ipynb`, replacing random construction with uniform sampling from the sub-400 archive. It deliberately performs **no neural training**.

In [ ]:
# Imports and Project Paths

%matplotlib inline

from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np

from ramsey import (
    RArchiveConstruction,
    RGraph,
    RProblem,
    RSearchState,
    RSQLiteArchive,
)
from ramsey.RVertexBalanceAction import (
    apply_vertex_balance_transfer,
    vertex_balance_energy,
    vertex_color_imbalances,
)
from ramsey.RVertexBalancePolicy import (
    RVertexBalancePolicyConfig,
    select_greedy_vertex_balance_action
)

project_root = Path.cwd().resolve()

if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "ramsey").is_dir():
    raise RuntimeError(
        "Run this notebook from the RamseyNumber root or notebooks directory."
    )

In [ ]:
# Experiment Configuration

RANDOM_SEED = 202_608_054
N_VERTICES = 43

NUMBER_OF_RUNS = 10
MAX_VERTEX_BALANCE_ACTIONS = 128
REPORT_INTERVAL = 10

ARCHIVE_SCORE_LIMIT = 399

DATABASE_PATH = (
    project_root
    / "data"
    / "ramsey_colorings.sqlite3"
)


DANGER_DECAY = 0.25
USE_DANGER_REWARD = True

prioritize_balance = True

In [ ]:
# Runtime, Problem, Graph, Archive, and Vertex-Balance Policy

rng = np.random.default_rng(
    RANDOM_SEED
)

graph_start = perf_counter()

problem = RProblem.r55(
    n_vertices=N_VERTICES,
)

graph = RGraph(
    problem
)

graph_elapsed = (
    perf_counter()
    - graph_start
)

existing_archive = globals().get(
    "archive"
)

if existing_archive is not None:
    existing_archive.close()

archive = RSQLiteArchive(
    DATABASE_PATH
)

eligible_archive_count = (
    archive.coloring_count_in_score_range(
        maximum_score=ARCHIVE_SCORE_LIMIT,
        graph=graph,
    )
)

if eligible_archive_count == 0:
    archive.close()

    raise RuntimeError(
        "Archive contains no eligible K43 colorings."
    )

construction = RArchiveConstruction(
    archive=archive,
    rng=rng,
    maximum_score=ARCHIVE_SCORE_LIMIT,
)

vertex_balance_policy_config = (
    RVertexBalancePolicyConfig(
        use_danger_reward=USE_DANGER_REWARD,
        danger_decay=DANGER_DECAY,
        prioritize_balance=True,
    )
)

print(
    "Problem:",
    problem,
)

print(
    "Edges:",
    f"{graph.number_of_edges:,}",
)

print(
    "K5s:",
    f"{graph.subgraph_index(5).clique_count:,}",
)

print(
    "Graph construction:",
    f"{graph_elapsed:.3f} seconds",
)

print(
    "Database:",
    DATABASE_PATH.resolve(),
)

print(
    "Construction:",
    construction.name,
)

print(
    "Eligible archive colorings:",
    eligible_archive_count,
)

print(
    "Archive best:",
    archive.best_score(graph),
)

print(
    "Runs:",
    NUMBER_OF_RUNS,
)

print(
    "Maximum vertex-balance actions per run:",
    MAX_VERTEX_BALANCE_ACTIONS,
)

print(
    "Use danger reward:",
    vertex_balance_policy_config.use_danger_reward,
)

print(
    "Danger decay:",
    vertex_balance_policy_config.danger_decay,
)

print(
    "Prioritize balance:",
    vertex_balance_policy_config.prioritize_balance,
)

In [ ]:
# Run the Greedy Vertex Balance Experiment

run_results = []

all_exact_rewards = []
all_objective_rewards = []
all_balance_rewards = []

experiment_start = perf_counter()

for run_number in range(NUMBER_OF_RUNS):
    seed_coloring = construction.construct(
        graph
    )

    seed_record = construction.last_record

    state = RSearchState(
        seed_coloring
    )

    initial_score = state.score

    initial_balance_energy = (
        vertex_balance_energy(state)
    )

    initial_imbalances = (
        vertex_color_imbalances(state)
    )

    best_score = state.score
    best_coloring = state.coloring_snapshot()

    score_trajectory = [
        state.score
    ]

    balance_trajectory = [
        initial_balance_energy
    ]

    seen_states = {
        state.colors.tobytes()
    }

    repeated_state = False
    balance_exhausted = False

    run_start = perf_counter()

    for action_number in range(
        MAX_VERTEX_BALANCE_ACTIONS
    ):
        if state.score == 0:
            break

        imbalances = (
            vertex_color_imbalances(state)
        )

        # The current balance action requires
        # both a blue-heavy donor and a
        # red-heavy recipient.
        if (
            not np.any(imbalances > 0)
            or not np.any(imbalances < 0)
        ):
            balance_exhausted = True
            break

        selection = (
            select_greedy_vertex_balance_action(
                state=state,
                config=(
                    vertex_balance_policy_config
                ),
            )
        )

        predicted_reward = (
            selection.exact_reward
        )

        before_balance_energy = (
            vertex_balance_energy(state)
        )

        actual_reward = (
            apply_vertex_balance_transfer(
                state,
                selection.transfer,
            )
        )

        after_balance_energy = (
            vertex_balance_energy(state)
        )

        actual_balance_reward = (
            before_balance_energy
            - after_balance_energy
        )

        if actual_reward != predicted_reward:
            raise RuntimeError(
                "Predicted and actual Ramsey "
                "rewards disagree: "
                f"{predicted_reward} versus "
                f"{actual_reward}."
            )

        if (
            actual_balance_reward
            != selection.balance_reward
        ):
            raise RuntimeError(
                "Predicted and actual balance "
                "rewards disagree: "
                f"{selection.balance_reward} "
                "versus "
                f"{actual_balance_reward}."
            )

        all_exact_rewards.append(
            actual_reward
        )

        all_objective_rewards.append(
            selection.objective_reward
        )

        all_balance_rewards.append(
            actual_balance_reward
        )

        score_trajectory.append(
            state.score
        )

        balance_trajectory.append(
            after_balance_energy
        )

        if state.score < best_score:
            best_score = state.score
            best_coloring = (
                state.coloring_snapshot()
            )

        state_key = (
            state.colors.tobytes()
        )

        if state_key in seen_states:
            repeated_state = True
            break

        seen_states.add(
            state_key
        )

        step_number = (
            action_number + 1
        )

        if (
            step_number
            % REPORT_INTERVAL
            == 0
        ):
            current_imbalances = (
                vertex_color_imbalances(
                    state
                )
            )

            print(
                f"Run {run_number:2d} | "
                f"archive_id="
                f"{seed_record.coloring_id:6d} | "
                f"action={step_number:3d} | "
                f"score={state.score:4d} | "
                f"best={best_score:4d} | "
                f"exact={actual_reward:+4d} | "
                f"balance="
                f"{after_balance_energy:5d} | "
                f"balance_reward="
                f"{actual_balance_reward:+4d} | "
                f"max_imbalance="
                f"{np.abs(current_imbalances).max():2d}"
            )

    run_elapsed = (
        perf_counter()
        - run_start
    )

    final_imbalances = (
        vertex_color_imbalances(state)
    )

    final_balance_energy = (
        vertex_balance_energy(state)
    )

    actions_completed = (
        len(score_trajectory) - 1
    )

    run_result = {
        "run": run_number,
        "archive_coloring_id": (
            seed_record.coloring_id
        ),
        "initial_score": initial_score,
        "final_score": state.score,
        "best_score": best_score,
        "best_coloring": best_coloring,
        "initial_balance_energy": (
            initial_balance_energy
        ),
        "final_balance_energy": (
            final_balance_energy
        ),
        "initial_max_imbalance": int(
            np.abs(
                initial_imbalances
            ).max()
        ),
        "final_max_imbalance": int(
            np.abs(
                final_imbalances
            ).max()
        ),
        "actions_completed": (
            actions_completed
        ),
        "balance_exhausted": (
            balance_exhausted
        ),
        "repeated_state": (
            repeated_state
        ),
        "elapsed": run_elapsed,
        "trajectory": np.asarray(
            score_trajectory,
            dtype=np.int32,
        ),
        "balance_trajectory": (
            np.asarray(
                balance_trajectory,
                dtype=np.int32,
            )
        ),
    }

    run_results.append(
        run_result
    )

    print(
        f"Run {run_number:2d} COMPLETE | "
        f"initial={initial_score:4d} | "
        f"final={state.score:4d} | "
        f"best={best_score:4d} | "
        f"balance="
        f"{initial_balance_energy}"
        f"->{final_balance_energy} | "
        f"max_imbalance="
        f"{np.abs(initial_imbalances).max()}"
        f"->{np.abs(final_imbalances).max()} | "
        f"actions={actions_completed:3d} | "
        f"exhausted={balance_exhausted}"
    )

experiment_elapsed = (
    perf_counter()
    - experiment_start
)

In [ ]:
# Experiment Summary

initial_scores = np.asarray(
    [result["initial_score"] for result in run_results],
    dtype=np.int32,
)
final_scores = np.asarray(
    [result["final_score"] for result in run_results],
    dtype=np.int32,
)
best_scores = np.asarray(
    [result["best_score"] for result in run_results],
    dtype=np.int32,
)

print("Runs:", len(run_results))
print("Mean initial score:", f"{initial_scores.mean():.2f}")
print("Mean final score:", f"{final_scores.mean():.2f}")
print("Mean best score:", f"{best_scores.mean():.2f}")
print("Minimum best score:", int(best_scores.min()))
print("Mean best reduction:", f"{(initial_scores - best_scores).mean():.2f}")
print("Improved runs:", f"{np.count_nonzero(best_scores < initial_scores)}/{len(run_results)}")
print("Repeated-state runs:", sum(result["repeated_state"] for result in run_results))
print("Total time:", f"{experiment_elapsed:.2f}s")
print("Mean run time:", f"{np.mean([result['elapsed'] for result in run_results]):.2f}s")

In [ ]:
# Vertex-Balance Action Statistics

exact_rewards = np.asarray(
    all_exact_rewards,
    dtype=np.int32,
)

objective_rewards = np.asarray(
    all_objective_rewards,
    dtype=np.float64,
)

balance_rewards = np.asarray(
    all_balance_rewards,
    dtype=np.int32,
)

print(
    "Balance transfers:",
    len(exact_rewards),
)

if len(exact_rewards):
    print(
        "Positive Ramsey reward:",
        f"{np.mean(exact_rewards > 0):.1%}",
    )

    print(
        "Zero Ramsey reward:",
        f"{np.mean(exact_rewards == 0):.1%}",
    )

    print(
        "Negative Ramsey reward:",
        f"{np.mean(exact_rewards < 0):.1%}",
    )

    print(
        "Mean Ramsey reward:",
        f"{exact_rewards.mean():+.2f}",
    )

    print(
        "Mean danger reward:",
        f"{objective_rewards.mean():+.3f}",
    )

    print(
        "Mean balance reward:",
        f"{balance_rewards.mean():+.2f}",
    )

print(
    "Balance-exhausted runs:",
    sum(
        result["balance_exhausted"]
        for result in run_results
    ),
)

In [ ]:
# Plot Score Trajectories

fig, axis = plt.subplots(
    figsize=(11, 6),
    constrained_layout=True,
)

for result in run_results:
    trajectory = result["trajectory"]

    axis.plot(
        np.arange(len(trajectory)),
        trajectory,
        alpha=0.60,
        linewidth=1.4,
    )

axis.set_xlabel("Vertex Balance Action")
axis.set_ylabel("Monochromatic K5 score")
axis.set_title("Greedy Vertex Balance Action Trajectories from Sub-400 Archive Seeds")
axis.grid(linestyle="--", alpha=0.3)

plt.show()

In [ ]:
# Release the SQLite Connection

archive.close()
print("Archive closed.")

In [9]:
# Independently audit the vertex-degree calculation.

manual_blue_degrees = np.zeros(
    N_VERTICES,
    dtype=np.int32,
)

manual_red_degrees = np.zeros(
    N_VERTICES,
    dtype=np.int32,
)

for edge_index, (u, v) in enumerate(
    graph.edges
):
    u = int(u)
    v = int(v)

    if state.colors[edge_index] == 1:
        manual_blue_degrees[u] += 1
        manual_blue_degrees[v] += 1
    else:
        manual_red_degrees[u] += 1
        manual_red_degrees[v] += 1

manual_imbalances = (
    manual_blue_degrees
    - manual_red_degrees
)

manual_energy = int(
    np.sum(
        manual_imbalances ** 2
    )
)

print(
    "Blue edges:",
    int(np.count_nonzero(state.colors == 1)),
)

print(
    "Red edges:",
    int(np.count_nonzero(state.colors == 0)),
)

print(
    "Blue degrees:",
    np.sort(manual_blue_degrees),
)

print(
    "Red degrees:",
    np.sort(manual_red_degrees),
)

print(
    "Imbalances:",
    np.sort(manual_imbalances),
)

print(
    "Manual energy:",
    manual_energy,
)

print(
    "Function energy:",
    vertex_balance_energy(state),
)

print(
    "Functions agree:",
    np.array_equal(
        manual_imbalances,
        vertex_color_imbalances(state),
    ),
)

print(
    "Degree sanity check:",
    np.all(
        manual_blue_degrees
        + manual_red_degrees
        == 42
    ),
)

print(
    "Handshake sanity check:",
    manual_blue_degrees.sum(),
    "==",
    2 * np.count_nonzero(
        state.colors == 1
    ),
)

Blue edges: 452
Red edges: 451
Blue degrees: [21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21
 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 22]
Red degrees: [20 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21
 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21 21]
Imbalances: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 2]
Manual energy: 4
Function energy: 4
Functions agree: True
Degree sanity check: True
Handshake sanity check: 904 == 904
